# M5 State-Level Meta-Set Creation
**Depends on** `aggregate_M5_to_state()` having already been run (`data/M5/processed/m5_state_aggregated.pkl` must exist).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
DATA_PATH = PROJECT_ROOT / 'data' / 'M5' / 'processed' / 'm5_state_aggregated.pkl'
SAVE_DIR = PROJECT_ROOT / 'data' / 'M5' / 'meta_sets'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

m5_df = pd.read_pickle(DATA_PATH)
print(f"Loaded {len(m5_df)} rows, {m5_df['traj_id'].nunique()} state-item series "
      f"across {m5_df['state_id'].nunique()} states")

In [ ]:
# Filter to a single state (CA) so each item maps to exactly one series 
STATE_ID = 'CA'

m5_df = m5_df[m5_df['state_id'] == STATE_ID].copy()
print(f"After filtering to state={STATE_ID}: {len(m5_df)} rows, "
      f"{m5_df['traj_id'].nunique()} series (should equal item count for this state)")

## Split scheme

In [ ]:
SEED = 42
ITEM_COL = 'traj_id'   # state_id + "_" + item_id; effectively item-unique once filtered to one state
TIME_COL = 'time_idx'

TRAIN_START, TRAIN_END = 1, 1857
VAL_START, VAL_END = 1774, 1885
TEST_START, TEST_END = 1802, 1913

TRAIN_ITEM_COUNT = 50
VAL_ITEM_COUNT = 500
TEST_ITEM_COUNT = 1000

DATE_TAG = '0308'
VT_DATE_TAG = 'volfilt0508'  

In [ ]:
np.random.seed(SEED)

test_pool = set(m5_df.loc[(m5_df[TIME_COL] >= TEST_START) & (m5_df[TIME_COL] <= TEST_END), ITEM_COL].unique())
val_pool = set(m5_df.loc[(m5_df[TIME_COL] >= VAL_START) & (m5_df[TIME_COL] <= VAL_END), ITEM_COL].unique())
train_pool_period = set(m5_df.loc[(m5_df[TIME_COL] >= TRAIN_START) & (m5_df[TIME_COL] <= TRAIN_END), ITEM_COL].unique())

print(f"Pool sizes before exclusion: test={len(test_pool)}, val={len(val_pool)}, train={len(train_pool_period)}")

# Test reserved first.
if len(test_pool) < TEST_ITEM_COUNT:
    raise ValueError(f"Test pool too small: {len(test_pool)} < {TEST_ITEM_COUNT}")
test_items = np.random.choice(sorted(test_pool), size=TEST_ITEM_COUNT, replace=False)

# Val sampled from what's left after excluding test items.
val_pool_avail = val_pool - set(test_items)
if len(val_pool_avail) < VAL_ITEM_COUNT:
    raise ValueError(f"Val pool too small after excluding test items: {len(val_pool_avail)} < {VAL_ITEM_COUNT}")
val_items = np.random.choice(sorted(val_pool_avail), size=VAL_ITEM_COUNT, replace=False)

# Train pool excludes both val and test items entirely (fully disjoint).
train_pool = train_pool_period - (set(val_items) | set(test_items))
print(f"Disjoint pool sizes: test={len(test_items)}, val={len(val_items)}, train={len(train_pool)}")

shuffled_train_pool = np.random.permutation(sorted(train_pool)).tolist()
if len(shuffled_train_pool) < TRAIN_ITEM_COUNT:
    raise ValueError(f"Train pool too small: {len(shuffled_train_pool)} < {TRAIN_ITEM_COUNT}")

In [ ]:
QUERY_WINDOW_DAYS = 28
QUERY_VOLUME_MIN = 10  

def _query_window_totals(items, window_end):
    window_start = window_end - QUERY_WINDOW_DAYS + 1
    win = m5_df[
        (m5_df[ITEM_COL].isin(items)) &
        (m5_df[TIME_COL] >= window_start) & (m5_df[TIME_COL] <= window_end)
    ]
    return win.groupby(ITEM_COL)['unit_sales'].sum()

replacement_pool = set(shuffled_train_pool[TRAIN_ITEM_COUNT:])

def _replace_low_volume(items, window_end, label):
    totals = _query_window_totals(items, window_end)
    low_volume = set(totals[totals < QUERY_VOLUME_MIN].index)
    keep = [i for i in items if i not in low_volume]

    cand_totals = _query_window_totals(sorted(replacement_pool), window_end)
    cand_totals = cand_totals[cand_totals >= QUERY_VOLUME_MIN]
    replacements = np.random.choice(cand_totals.index.to_numpy(), size=len(low_volume), replace=False)
    replacement_pool.difference_update(replacements)

    new_items = np.array(keep + list(replacements))
    print(f"{label}: replaced {len(low_volume)}/{len(items)} low-volume items "
          f"(<{QUERY_VOLUME_MIN} units in final {QUERY_WINDOW_DAYS}d query window) "
          f"from the unused train_pool tail -> {len(new_items)} items, "
          f"{len(replacement_pool)} replacement candidates left")
    return new_items

val_items = _replace_low_volume(val_items, VAL_END, "Val")
test_items = _replace_low_volume(test_items, TEST_END, "Test")

assert (_query_window_totals(val_items, VAL_END) < QUERY_VOLUME_MIN).sum() == 0
assert (_query_window_totals(test_items, TEST_END) < QUERY_VOLUME_MIN).sum() == 0
assert not (set(val_items) & set(shuffled_train_pool[:TRAIN_ITEM_COUNT]))
assert not (set(test_items) & set(shuffled_train_pool[:TRAIN_ITEM_COUNT]))
assert not (set(val_items) & set(test_items))

In [ ]:
selected_items = shuffled_train_pool[:TRAIN_ITEM_COUNT]
train_df = m5_df[
    (m5_df[ITEM_COL].isin(selected_items)) &
    (m5_df[TIME_COL] >= TRAIN_START) & (m5_df[TIME_COL] <= TRAIN_END)
].copy()

n_actual = train_df[ITEM_COL].nunique()
if n_actual != TRAIN_ITEM_COUNT:
    raise ValueError(f"Expected {TRAIN_ITEM_COUNT} items with data in train period, got {n_actual}")

save_path = SAVE_DIR / f'Seed{SEED}_{STATE_ID}_itemBased_meta_train_cumulative_{TRAIN_ITEM_COUNT}items_{DATE_TAG}.pkl'
train_df.to_pickle(save_path)

In [ ]:
val_df = m5_df[
    (m5_df[ITEM_COL].isin(val_items)) &
    (m5_df[TIME_COL] >= VAL_START) & (m5_df[TIME_COL] <= VAL_END)
].copy()
n_actual = val_df[ITEM_COL].nunique()
if n_actual != VAL_ITEM_COUNT:
    raise ValueError(f"Val: expected {VAL_ITEM_COUNT} items, got {n_actual}")

save_path = SAVE_DIR / f'Seed{SEED}_{STATE_ID}_itemBased_meta_val_{VAL_ITEM_COUNT}items_{VT_DATE_TAG}.pkl'
val_df.to_pickle(save_path)
print(f"Val: saved {len(val_df)} rows, {n_actual} series -> {save_path.name}")

In [ ]:
test_df = m5_df[
    (m5_df[ITEM_COL].isin(test_items)) &
    (m5_df[TIME_COL] >= TEST_START) & (m5_df[TIME_COL] <= TEST_END)
].copy()
n_actual = test_df[ITEM_COL].nunique()
if n_actual != TEST_ITEM_COUNT:
    raise ValueError(f"Test: expected {TEST_ITEM_COUNT} items, got {n_actual}")

save_path = SAVE_DIR / f'Seed{SEED}_{STATE_ID}_itemBased_meta_test_{TEST_ITEM_COUNT}items_{VT_DATE_TAG}.pkl'
test_df.to_pickle(save_path)
print(f"Test: saved {len(test_df)} rows, {n_actual} series -> {save_path.name}")